# Handwritten digit classification with MNIST

This notebook compares a fully connected neural network with a convolutional neural network (CNN) on the 28 × 28 grayscale MNIST digits. It walks through loading and preparing the data, training both models, and evaluating their predictions on the held-out test set.

The test set is reserved for final evaluation; use the validation results to make training decisions. MNIST is a learning benchmark, not evidence that a model will work equally well on other handwriting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

SEED = 42
tf.keras.utils.set_random_seed(SEED)
print(f"TensorFlow version: {tf.__version__}")

## Load and prepare the data

Keras downloads MNIST the first time this cell runs. Pixel values are converted to `float32` and scaled to `[0, 1]`. The extra channel dimension is required by convolutional layers; the dense model will flatten each image internally.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = (x_train.astype("float32") / 255.0)[..., np.newaxis]
x_test = (x_test.astype("float32") / 255.0)[..., np.newaxis]

print(f"Training images: {x_train.shape}; labels: {y_train.shape}")
print(f"Test images:     {x_test.shape}; labels: {y_test.shape}")
print(f"Pixel range:     [{x_train.min():.1f}, {x_train.max():.1f}]")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for index, ax in enumerate(axes.flat):
    ax.imshow(x_train[index].squeeze(), cmap="gray")
    ax.set_title(f"Label: {y_train[index]}")
    ax.axis("off")
fig.suptitle("Example training images")
fig.tight_layout()
plt.show()

## Baseline: fully connected network

A dense network treats the image as a vector, so it does not directly use the spatial relationships between neighboring pixels. Early stopping monitors validation loss and restores the best weights.

In [ ]:
def make_callbacks():
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True
        )
    ]

tf.keras.utils.set_random_seed(SEED)
dense_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ],
    name="dense_baseline",
)
dense_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
dense_model.summary()

In [ ]:
dense_history = dense_model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    callbacks=make_callbacks(),
    verbose=2,
)

### Inspect the dense model's learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, title in zip(axes, ["accuracy", "loss"], ["Accuracy", "Loss"]):
    ax.plot(dense_history.history[metric], label="Training")
    ax.plot(dense_history.history[f"val_{metric}"], label="Validation")
    ax.set(title=title, xlabel="Epoch", ylabel=title)
    ax.legend()
fig.tight_layout()
plt.show()

## CNN: learn local image features

Convolution and pooling layers learn patterns such as edges and strokes while preserving the image's spatial structure. The final softmax layer returns one probability per digit class.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
cnn_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(32, kernel_size=3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, kernel_size=3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ],
    name="cnn",
)
cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
cnn_model.summary()

In [ ]:
cnn_history = cnn_model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    callbacks=make_callbacks(),
    verbose=2,
)

## Final evaluation

Evaluate each model once on the held-out test set. Accuracy gives an overall score; the per-class report and confusion matrix show which digits are being confused. The test data should not be used to tune the models.

In [ ]:
def evaluate_model(model, name):
    probabilities = model.predict(x_test, verbose=0)
    predictions = probabilities.argmax(axis=1)
    test_accuracy = np.mean(predictions == y_test)
    test_loss = tf.keras.losses.sparse_categorical_crossentropy(
        y_test, probabilities
    ).numpy().mean()
    print(f"{name} — test accuracy: {test_accuracy:.4f}; test loss: {test_loss:.4f}")
    print(classification_report(y_test, predictions, digits=3, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predictions,
        cmap="Blues",
        colorbar=False,
    )
    plt.title(f"{name} confusion matrix")
    plt.tight_layout()
    plt.show()
    return test_accuracy, predictions, probabilities

dense_accuracy, dense_predictions, dense_probabilities = evaluate_model(
    dense_model, "Dense baseline"
)
cnn_accuracy, cnn_predictions, cnn_probabilities = evaluate_model(cnn_model, "CNN")
print(f"\nAccuracy difference (CNN - dense): {cnn_accuracy - dense_accuracy:+.4f}")

### Inspect sample CNN predictions

In [ ]:
incorrect = np.flatnonzero(cnn_predictions != y_test)
if len(incorrect) == 0:
    print("No incorrect predictions in the test set.")
else:
    sample_indices = incorrect[:10]
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for ax, index in zip(axes.flat, sample_indices):
        confidence = cnn_probabilities[index, cnn_predictions[index]]
        ax.imshow(x_test[index].squeeze(), cmap="gray")
        ax.set_title(
            f"True: {y_test[index]} | Pred: {cnn_predictions[index]}\n"
            f"Confidence: {confidence:.2f}"
        )
        ax.axis("off")
    for ax in axes.flat[len(sample_indices):]:
        ax.axis("off")
    fig.suptitle("Examples the CNN misclassified")
    fig.tight_layout()
    plt.show()

## Takeaways

- The dense network is a useful baseline, but flattening discards the local layout of pixels.
- CNNs can use spatial structure to recognize visual patterns; compare the measured results rather than assuming one architecture is always better.
- Review errors and class-level metrics before using a classifier outside this benchmark. Do not treat confidence as a guarantee of correctness.